# Qwen CT Inference (Video Input)

In [ ]:
import os
os.environ["DASHSCOPE_API_KEY"] = "<YOUR_DASHSCOPE_API_KEY>"

import base64
import io
import json
import subprocess
import tempfile
from typing import List

import numpy as np
import pandas as pd
import pydicom
import PIL.Image
import imageio
from IPython.display import display, Markdown
from openai import OpenAI

# ── Data paths ────────────────────────────────────────────────────────────────
DATA_PATH       = "../data/images/dev"
PRELIM_DIR      = "../data/dev_preliminary_report"
EDITS_JSON_PATH = "../data/dev_edits.json"

# ── Model / sampling ─────────────────────────────────────────────────────────
QWEN_MODEL         = "qwen3.5-plus"
MAX_SLICE          = 50    # slices sampled per series
VIDEO_FPS          = 10    # frames per second for generated MP4
# Base64 adds ~33% overhead; keep source file ≤7.5 MB so encoded payload ≤10 MB
MAX_VIDEO_SIZE_BYTES = int(7.5 * 1024 * 1024)

print(f"Data path          : {DATA_PATH}  exists={os.path.exists(DATA_PATH)}")
print(f"Prelim dir         : {PRELIM_DIR}  exists={os.path.exists(PRELIM_DIR)}")
print(f"Edits JSON         : {EDITS_JSON_PATH}  exists={os.path.exists(EDITS_JSON_PATH)}")
print(f"QWEN_MODEL         : {QWEN_MODEL}")
print(f"MAX_SLICE          : {MAX_SLICE}")
print(f"VIDEO_FPS          : {VIDEO_FPS}")
print(f"MAX_VIDEO_SIZE_BYTES: {MAX_VIDEO_SIZE_BYTES / (1024*1024):.1f} MB")

Data path          : /home/NETID/zhaoyis/code/RADAR_for_organizers/data/images/dev  exists=True
Prelim dir         : /home/NETID/zhaoyis/code/RADAR_for_organizers/data/dev_preliminary_report  exists=True
Edits JSON         : /home/NETID/zhaoyis/code/RADAR_for_organizers/data/dev_edits.json  exists=True
QWEN_MODEL         : qwen3.5-plus
MAX_SLICE          : 50
VIDEO_FPS          : 10
MAX_VIDEO_SIZE_BYTES: 7.5 MB


In [2]:
assert os.getenv("DASHSCOPE_API_KEY"), "Please set DASHSCOPE_API_KEY."
DASHSCOPE_BASE_URL = os.getenv(
    "DASHSCOPE_BASE_URL",
    "https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)
client = OpenAI(api_key=os.getenv("DASHSCOPE_API_KEY"), base_url=DASHSCOPE_BASE_URL)
print(f"Qwen client initialized at: {DASHSCOPE_BASE_URL}")

Qwen client initialized at: https://dashscope-intl.aliyuncs.com/compatible-mode/v1


In [3]:
with open(EDITS_JSON_PATH) as f:
    edits_data = json.load(f)

cases = []
for item in edits_data:
    case_id = item["case_id"]
    prelim_path = os.path.join(PRELIM_DIR, f"{case_id}.txt")
    with open(prelim_path) as pf:
        preliminary_report = pf.read().strip()
    cases.append({
        "case_id": case_id,
        "clinical_indication": item["clinical_indication"],
        "preliminary_report": preliminary_report,
        "edits": item["edits"],
    })

total_edits = sum(len(c["edits"]) for c in cases)
print(f"Loaded {len(cases)} cases, {total_edits} edits total")
print()
c0 = cases[0]
print(f"First case preview:")
print(f"  case_id            : {c0['case_id']}")
print(f"  clinical_indication: {c0['clinical_indication']}")
print(f"  prelim (first 200) : {c0['preliminary_report'][:200]}")
print(f"  edits              : {[e['edit_id'] for e in c0['edits']]}")

Loaded 20 cases, 184 edits total

First case preview:
  case_id            : 10008
  clinical_indication: eval for small bowel fistula, perinephric hematoma resolved, but now with air possibly communicating with small bowel
  prelim (first 200) : PRELIMINARY INPATIENT FINDINGS/IMPRESSION
 
In response to the clinical question,
 
Similar appearance of perinephric fluid collection, now with mild increase in layering fluid at the superior most as
  edits              : ['10008_01', '10008_02', '10008_03', '10008_04', '10008_05', '10008_06', '10008_07']


In [4]:
print(f"{'case_id':>8}  {'#edits':>6}  series")
print("-" * 60)
for c in cases:
    case_dir = os.path.join(DATA_PATH, str(c["case_id"]))
    series = sorted([
        s for s in os.listdir(case_dir)
        if os.path.isdir(os.path.join(case_dir, s))
    ]) if os.path.exists(case_dir) else []
    print(f"{c['case_id']:>8}  {len(c['edits']):>6}  {series}")

 case_id  #edits  series
------------------------------------------------------------
   10008       7  ['ABD_PEL', 'COR_BODY', 'SAG_BODY', 'SCOUT']
   10009      11  ['ABD_PEL', 'COR_BODY', 'SAG_BODY', 'SCOUT']
   10011       6  ['AXIAL_2.5mm', 'CHEST_MIP', 'COR_AORTA', 'LUNG_2.5mm', 'SAG_AORTA', 'Scout']
   10014      14  ['AXIAL_2.5mm', 'CCTA_Center_Phase', 'CHEST_MIP', 'COR_AORTA', 'LUNG_2.5mm', 'SAG_AORTA', 'Scout']
   10019       9  ['AXIAL_MIPS', 'Bone_Algorithm_Chest', 'CORONAL', 'Chest_Survey_without_Contrast', 'Chest_without_Contrast', 'SAGITTAL']
   10020      11  ['C-A-P,_W_IV', 'CHEST_MIP', 'COR_BODY', 'LUNG_ALG', 'SAG_BODY', 'SCOUT']
   10022      14  ['Axial_3mm_VNC', 'Axial_DE_Venous', 'Coronal_MIP_Abdomen_F_0.6', 'Coronal_Venous', 'DE_Axial_CVA_9_F_0.6', 'DE_PP_VNC_Mesenteric_A_90kV', 'DE_PP_VNC_Mesenteric_B_Sn150kV', 'Iodine_mapping_3.0mm', 'Monitoring', 'Sagittal_MIP_Abdomen_F_0.6', 'Scout_AP', 'Scout_LAT']
   10023       6  ['CHEST_MIP', 'COR_BODY', 'LUNG_ALG', 'SAG

In [5]:
def load_ct_series_from_directory(
    series_path: str,
    max_slices: int = MAX_SLICE,
    verbose: bool = True,
) -> List[np.ndarray]:
    """Load a CT series, returning up to max_slices uniformly sampled slices."""
    all_files = [
        os.path.join(series_path, f)
        for f in os.listdir(series_path)
        if not f.startswith(".") and not f.endswith((".txt", ".json", ".xml", ".csv"))
    ] if os.path.isdir(series_path) else []

    if verbose:
        print(f"Found {len(all_files)} files to check")

    dicom_instances, failed = [], 0
    for fp in all_files:
        try:
            dcm = pydicom.dcmread(fp, stop_before_pixels=True)
            if getattr(dcm, "Modality", None) == "CT" and hasattr(dcm, "InstanceNumber"):
                dicom_instances.append((fp, int(dcm.InstanceNumber)))
            else:
                failed += 1
        except Exception:
            failed += 1

    if not dicom_instances:
        raise ValueError(f"No CT DICOM files found in {series_path}")

    dicom_instances.sort(key=lambda x: x[1])
    if verbose:
        print(f"Found {len(dicom_instances)} CT slices ({failed} non-CT/failed)")

    if len(dicom_instances) > max_slices:
        idxs = [int(round(i / max_slices * (len(dicom_instances) - 1))) for i in range(1, max_slices + 1)]
        dicom_instances = [dicom_instances[i] for i in idxs]
        if verbose:
            print(f"Sampled down to {len(dicom_instances)} slices")

    slices = []
    for dcm_path, _ in dicom_instances:
        try:
            dcm = pydicom.dcmread(dcm_path)
            slices.append(pydicom.pixels.apply_rescale(dcm.pixel_array, dcm))
        except Exception as e:
            print(f"Error reading {dcm_path}: {e}")

    if verbose:
        print(f"Loaded {len(slices)} slices successfully")
    return slices


def norm(ct: np.ndarray, lo: float, hi: float) -> np.ndarray:
    ct = np.clip(ct, lo, hi).astype(np.float32)
    return (ct - lo) / (hi - lo) * 255.0


def window(ct: np.ndarray) -> np.ndarray:
    """Three-channel windowing (wide / soft-tissue / brain)."""
    clips = [(-1024, 1024), (-135, 215), (0, 80)]
    return np.stack([norm(ct, lo, hi) for lo, hi in clips], axis=-1)


def preprocess_slices(raw_slices: List[np.ndarray]) -> List[np.ndarray]:
    return [np.round(window(s), 0).astype(np.uint8) for s in raw_slices]


def slices_to_mp4(frames: List[np.ndarray], fps: int = VIDEO_FPS) -> str:
    """
    Write uint8 RGB frames to a temporary MP4 file.
    Returns the temp file path (caller is responsible for deletion).
    """
    tmp = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    tmp.close()
    with imageio.get_writer(tmp.name, fps=fps, format="ffmpeg", codec="libx264",
                            output_params=["-pix_fmt", "yuv420p"]) as writer:
        for frame in frames:
            writer.append_data(frame)
    return tmp.name


def _compress_video(video_path: str, target_size_bytes: int) -> str:
    """
    Re-encode a video with ffmpeg bitrate control to fit within target_size_bytes.
    Returns the path to a new temp file (caller must delete it).
    """
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", video_path],
        capture_output=True, text=True, check=True,
    )
    duration = float(result.stdout.strip())
    video_bps = max(int((target_size_bytes * 8 * 0.92) / duration) - 64_000, 50_000)

    tmp = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    tmp.close()
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path,
         "-b:v", str(video_bps), "-maxrate", str(video_bps),
         "-bufsize", str(video_bps * 2), "-b:a", "64000", tmp.name],
        capture_output=True, check=True,
    )
    return tmp.name


def encode_video_data_url(video_path: str) -> str:
    """
    Base64-encode an MP4 for the DashScope video_url API.
    If the file exceeds MAX_VIDEO_SIZE_BYTES it is compressed first so the
    encoded payload stays under the 10 MB API limit.
    """
    size = os.path.getsize(video_path)
    tmp_path = None
    if size > MAX_VIDEO_SIZE_BYTES:
        print(f"  Video {size / (1024*1024):.2f} MB > limit — compressing...")
        tmp_path = _compress_video(video_path, MAX_VIDEO_SIZE_BYTES)
        print(f"  Compressed: {os.path.getsize(tmp_path) / (1024*1024):.2f} MB")
        path_to_encode = tmp_path
    else:
        path_to_encode = video_path
    try:
        with open(path_to_encode, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
    finally:
        if tmp_path:
            os.unlink(tmp_path)
    return f"data:video/mp4;base64,{b64}"


print("Helpers ready.")

Helpers ready.


In [6]:
# ── Configure which case/edit/series to inspect ───────────────────────────────
SELECTED_CASE_ID  = 10008     # any case_id listed above
SELECTED_EDIT_IDX = 0          # 0-based index into that case's edits list
SELECTED_SERIES   = "COR_BODY"  # series name to load

case = next(c for c in cases if c["case_id"] == SELECTED_CASE_ID)
edit = case["edits"][SELECTED_EDIT_IDX]

clinical_indication = case["clinical_indication"]
preliminary_report  = case["preliminary_report"]
suggested_edit      = edit["suggested_edit_text"]
edit_id             = edit["edit_id"]

case_path = os.path.join(DATA_PATH, str(SELECTED_CASE_ID))
available_series = sorted([
    s for s in os.listdir(case_path)
    if os.path.isdir(os.path.join(case_path, s))
])
series_path = os.path.join(case_path, SELECTED_SERIES)

print(f"Case ID          : {SELECTED_CASE_ID}")
print(f"Edit ID          : {edit_id}")
print(f"Clinical ind.    : {clinical_indication}")
print(f"Suggested edit   : {suggested_edit}")
print(f"Available series : {available_series}")
print(f"Using series     : {SELECTED_SERIES}  (path exists: {os.path.exists(series_path)})")
print()

# Load + preprocess DICOM
ct_volume_slices  = load_ct_series_from_directory(series_path, verbose=True)
normalized_slices = preprocess_slices(ct_volume_slices)
print(f"\nNormalized slices : {len(normalized_slices)}  shape: {normalized_slices[0].shape}")

# Build MP4
print("\nBuilding MP4...")
tmp_video_path = slices_to_mp4(normalized_slices, fps=VIDEO_FPS)
video_size_mb = os.path.getsize(tmp_video_path) / (1024 * 1024)
print(f"Temp video       : {tmp_video_path}")
print(f"Size             : {video_size_mb:.2f} MB  "
      f"(limit before encoding: {MAX_VIDEO_SIZE_BYTES / (1024*1024):.1f} MB)")

Case ID          : 10008
Edit ID          : 10008_01
Clinical ind.    : eval for small bowel fistula, perinephric hematoma resolved, but now with air possibly communicating with small bowel
Suggested edit   : A left renal angiomyolipoma measuring approximately 3.5 cm is identified.
Available series : ['ABD_PEL', 'COR_BODY', 'SAG_BODY', 'SCOUT']
Using series     : COR_BODY  (path exists: True)

Found 106 files to check
Found 106 CT slices (0 non-CT/failed)
Sampled down to 50 slices
Loaded 50 slices successfully

Normalized slices : 50  shape: (512, 512, 3)

Building MP4...


Multiple -pix_fmt options specified for stream 0, only the last option '-pix_fmt yuv420p' will be used.


Temp video       : /tmp/tmpjg16b7g_.mp4
Size             : 1.17 MB  (limit before encoding: 7.5 MB)


In [8]:
instruction = """
You are a highly rigorous and skeptical Attending Radiologist. Your task is to ensure the preliminary report is accurate and clinically actionable. You believe a good report focuses on findings relevant to the patient's presentation. You only "agree" with an edit if it is factually perfect and provides essential clinical value relative to the patient's condition.

You will be provided with:
- Clinical Indication: The specific reason the scan was ordered.
- Preliminary Radiology Report: The initial draft.
- CT Scan Volume (Ground Truth): The reference standard provided as a video.
- A Suggested Edit: A proposed change (Note: This may be \"synthetic\" or factually incorrect).

You must follow these steps in order:

Step 1: Internal Clinical Monologue (Reasoning)

Before providing labels, briefly answer these four questions:

1. Intent: What is the primary goal of this edit?
2. Accuracy: Does the Ground Truth (CT Scan) support this edit?
3. Indication Filtering: How relevant is this edit to the Clinical Indication? Is it a \"must-know\" for the referring physician, or an incidental finding?
4. Clinical Risk Assessment: If the edit is correct: What is the clinical penalty of omitting it? If the edit is wrong: What is the danger of including it?

Step 2: Agreement Decision

- agree: The edit is 100% accurate AND provides a clinically meaningful finding that was missing or wrong.
- partially agree: The edit is generally correct but nitpicky, contains minor inaccuracies, or describes information already implied in the report.
- disagree: The edit contains factual errors, misinterprets the imaging, or is redundant/unnecessary.

Step 3: Severity Assessment

- critical: Omitting/including the edit would cause immediate life-threatening harm or dangerous unnecessary intervention.
- moderate: Omitting/including the edit would delay non-emergent care or trigger unnecessary follow-up.
- negligible: No practical impact on patient management.

Step 4: Edit Type (Intention)

- correction: The edit intends to fix a wrong finding in the preliminary report.
- addition: The edit intends to add a missing finding to the preliminary report.
- clarification: The edit intends to refine an imprecise description already in the report.
"""

print("Instruction defined.")

Instruction defined.


In [9]:
def make_query(clinical_indication: str, preliminary_report: str, suggested_edit: str) -> str:
    return (
        f"Clinical Indication: {clinical_indication}\n"
        f"Preliminary Report\n{preliminary_report}\n"
        f"Suggested Edit: {suggested_edit}\n\n"
        "Your response must be strictly in JSON format:\n"
        "{\n"
        '  "agreement": "agree|partially agree|disagree",\n'
        '  "severity": "critical|moderate|negligible",\n'
        '  "suggested_edit_type": "correction|addition|clarification",\n'
        '  "explanation": "brief explanation"\n'
        "}"
    )


query_text   = make_query(clinical_indication, preliminary_report, suggested_edit)
video_data_url = encode_video_data_url(tmp_video_path)

# Clean up temp file now that it's encoded
os.unlink(tmp_video_path)
print("Temp file removed.")

user_content = [
    {"type": "video_url", "video_url": {"url": video_data_url}},
    {"type": "text",      "text": query_text},
]

response = client.chat.completions.create(
    model=QWEN_MODEL,
    messages=[
        {"role": "system", "content": instruction},
        {"role": "user",   "content": user_content},
    ],
    temperature=0.3,
    top_p=0.9,
    max_tokens=32768,
)

qwen_response = response.choices[0].message.content
if isinstance(qwen_response, list):
    qwen_response = "\n".join(
        part.get("text", "") for part in qwen_response if isinstance(part, dict)
    )
qwen_response = qwen_response or str(response)
display(Markdown(f"**Edit ID:** `{edit_id}`\n\n---\n\n{qwen_response}\n\n---"))

Temp file removed.


**Edit ID:** `10008_01`

---

{
  "agreement": "disagree",
  "severity": "moderate",
  "suggested_edit_type": "addition",
  "explanation": "The edit incorrectly diagnoses a left renal angiomyolipoma. The imaging demonstrates a perinephric collection with air, consistent with the clinical indication of a small bowel fistula, rather than an intrarenal fat-containing mass. This likely represents a misinterpretation of gas (low attenuation) as macroscopic fat. Including a false diagnosis of a renal tumor triggers unnecessary surveillance and patient anxiety, warranting a moderate severity rating, as the primary acute pathology is still reported in the preliminary text."
}

---

In [11]:
def is_valid_json_output(s: str) -> bool:
    if not isinstance(s, str) or s.startswith("ERROR:"):
        return False
    try:
        s = s.strip()
        data = json.loads(s[s.find("{"):s.rfind("}")+1])
        return data.get("agreement") is not None
    except Exception:
        return False


def extract_labels(s: str):
    """Returns (agreement, severity, edit_type) or (None, None, None)."""
    try:
        s = s.strip()
        data = json.loads(s[s.find("{"):s.rfind("}")+1])
        return (
            data.get("agreement"),
            data.get("severity"),
            data.get("suggested_edit_type"),
        )
    except Exception as e:
        print(f"JSON parse error: {e}")
    return None, None, None


print("Helpers ready.")
print(f"Valid JSON  : {is_valid_json_output(qwen_response)}")
print(f"Labels      : {extract_labels(qwen_response)}")

Helpers ready.
Valid JSON  : True
Labels      : ('disagree', 'moderate', 'addition')
